# Project 3: Semantic Segmentation

#### Maximum Points: 100

This project evaluates your understanding and implementation of a semantic segmentation pipeline. The overall project is divided into several sections, with specific marks assigned to each.


**Mark Distribution**:

<div align = "center">
    <table>
        <tr><td><h3>No.</h3></td> <td><h3>Section</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1.1.</h3></td> <td><h3> Dataset Class</h3></td> <td><h3>15</h3></td> </tr>
        <tr><td><h3>1.2</h3></td> <td><h3> Visualize dataset</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2.</h3></td> <td><h3>  Define Evaluation Metrics</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>3.</h3></td> <td><h3>  Model Definition</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>4.</h3></td> <td><h3>  Train</h3></td> <td><h3>15</h3></td> </tr>
        <tr><td><h3>5.</h3></td> <td><h3>  Inference</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>6.</h3></td> <td><h3> mIoU Score</h3></td> <td><h3>30</h3></td> </tr>
    </table>
        
</div>

---

Breakdown of mIoU Score on Test Dataset **(30 Points)**:

The mIoU Score section evaluates the quality of your segmentation model. Points are awarded based on the achieved mIoU (Mean Intersection over Union) score on the test dataset, as shown below:

<div align="center">
    <table>
        <tr><td><h3>No.</h3></td> <td><h3>mIoU Score Range</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1.</h3></td> <td><h3>>=65% and < 70%</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2.</h3></td> <td><h3>>= 70% and < 73%</h3></td> <td><h3>20</h3></td> </tr>
        <tr><td><h3>3.</h3></td> <td><h3>>= 73%</h3></td><td><h3>30</h3></td> </tr>
    </table>
</div>


---
<h2 style = "color: green;">Dataset Description </h2>

For this Semantic Segmentation Project, you will be working on the **[ICPR Retinal Blood Vessels Dataset](https://www.dropbox.com/scl/fi/tscgh3pxwzfvesnu6l6uv/icpr_prepared.zip?rlkey=8oay8sod3jc1hvwhgqvylaefr&st=udj92wmp&dl=1).**


The dataset consists of **268** train images and **112** test images in 2 classes (**veins** and **background**).

There is no separate validation dataset, you can use the given test dataset for validation.

The dataset is structured as follows:

```
icpr_prepared/
├── train_images/
│   ├── image_100.tif
│   ├── image_101.tif
│   └── ...
├── train_labels/
│   ├── image_100.tif
│   ├── image_101.tif
│   └── ...
├── test_images/
│   ├── image_121.tif
│   ├── image_122.tif
│   └── ...
└── test_labels/
    ├── image_121.tif
    ├── image_122.tif
    └── ...

```
---
Images and their corresponding binary masks are of `.tif` file format and are named with a unique `ImageId` as the filename.

Few samples are shown below:

<img src="https://www.dropbox.com/scl/fi/h0z30ptb8o6cyppfs8h4w/segformer_data_viz_grid_2.png?rlkey=tssqlts8o4kea3kf1um3tijjs&st=0sxunbha&dl=1" width="800" height="800">


**The notebook is divided into multiple grading sections. You have to write code, as mentioned for each section.  For other helper functions, you can write `.py` files and import them in the notebook. You have to submit the notebook along with `.py` files. Your submitted code must be runnable without any bug.**

In [17]:
# !pip install segmentation-models-pytorch torchinfo

In [18]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from dataclasses import dataclass
import cv2
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
import torch.optim as TorchOptimizers
import torch.amp
from torchinfo import summary as torch_summary
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import time
import wandb
import pandas as pd
import albumentations as A
import json
import math
from typing import Callable
from pathlib import Path
import random

In [19]:
# from google.colab import drive
# drive.mount('/content/drive')

In [20]:
# wandb.login()

In [21]:
@dataclass
class Environment:
    train_images_folder: str
    train_labels_folder: str
    val_images_folder: str
    val_labels_folder: str
    saved_weights_filepath: str
    training_output_folder: str
    device: str

    def fetch_train_filenames(self):
        names = np.array([f.name for f in Path(self.train_images_folder).iterdir()])
        if config.fraction >= .99:
            return names
        fraction_of_names = random.choices(names, k=math.ceil(len(names) * config.fraction))
        return fraction_of_names

    def fetch_val_filenames(self):
        return np.array([f.name for f in Path(self.val_images_folder).iterdir()])


env: Environment = None
""" Set to appropriate environment before training/inference """;

In [22]:
local_env = Environment(
    train_images_folder='data/train_images/',
    train_labels_folder='data/train_labels/',
    val_images_folder='data/test_images/',
    val_labels_folder='data/test_labels/',
    saved_weights_filepath='data_gen/best.pt',
    training_output_folder='data_gen/',
    device='cpu',
)
colab_home_folder = '/content/drive/My Drive/Colab Notebooks/retina-segmentation/'
colab_env = Environment(
    train_images_folder=colab_home_folder + 'data/train_images/',
    train_labels_folder=colab_home_folder + 'data/train_labels/',
    val_images_folder=colab_home_folder + 'data/test_images/',
    val_labels_folder=colab_home_folder + 'data/test_labels/',
    saved_weights_filepath=colab_home_folder + 'data_gen/best.pt',
    training_output_folder=colab_home_folder + 'data_gen/',
    device='cuda',
)

In [23]:
class Config:
    def __init__(self, training, verbose=False, debug=False):
        self.verbose = verbose
        self.training = training
        if self.training:
            os.makedirs(env.training_output_folder, exist_ok=True)

        self.num_classes = 1
        self.seed = 8675309
        self.batch_size = 32
        self.starting_learning_rate = 1e-4
        self.max_epochs = 400
        self.patience = 100
        self.num_workers = 9 if env.device == 'cuda' and not debug else 0
        self.pin_memory = self.num_workers > 0
        self.use_amp = env.device == 'cuda'
        self.original_image_width = 768
        self.original_image_height = 576
        self.image_width = 768
        self.image_height = 576
        self.fraction = 0.1 if debug else 1

        self.imagenet_mean_cpu_tensor = torch.tensor(imagenet_mean_array)
        self.imagenet_std_cpu_tensor = torch.tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_cpu_tensor = self.imagenet_mean_cpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_cpu_tensor = self.imagenet_std_cpu_tensor.view(3, 1, 1)
        self.imagenet_mean_gpu_tensor = gpu_tensor(imagenet_mean_array)
        self.imagenet_std_gpu_tensor = gpu_tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_gpu_tensor = self.imagenet_mean_gpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_gpu_tensor = self.imagenet_std_gpu_tensor.view(3, 1, 1)

        self.encoder_name = 'resnet34'
        self.encoder_weights = 'imagenet'

        torch.manual_seed(self.seed)
        self.generator = torch.Generator(device='cpu').manual_seed(self.seed)

        self.train_transforms = A.Compose([
            A.Resize(self.image_height, self.image_width),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.03,
                scale_limit=0.05,
                rotate_limit=10,
                interpolation=cv2.INTER_LINEAR,
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),
            A.OneOf([
                A.CLAHE(clip_limit=(1, 4), tile_grid_size=(8, 8), p=1.0),
                A.RandomBrightnessContrast(0.15, 0.15, p=1.0),
                A.RandomGamma(gamma_limit=(85, 115), p=1.0),
            ], p=0.5),
            A.Normalize(mean=imagenet_mean_tuple, std=imagenet_std_tuple),
            A.ToTensorV2(),
        ])

        self.val_transforms = A.Compose([
            A.Resize(self.image_height, self.image_width),
            A.Normalize(mean=imagenet_mean_tuple, std=imagenet_std_tuple),
            A.ToTensorV2(),
        ])

        self.test_transforms = A.Compose([
            A.Normalize(mean=imagenet_mean_tuple, std=imagenet_std_tuple),
            A.ToTensorV2(),
        ])


config: Config = None
""" Create and assign before training/inference """;

In [24]:
imagenet_mean_tuple = (0.485, 0.456, 0.406)
imagenet_std_tuple = (0.229, 0.224, 0.225)
imagenet_mean_array = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std_array = np.array([0.229, 0.224, 0.225], dtype=np.float32)

CLASS_COLORS = np.array([
    [  0,   0,   0], # 0: Black (Background)
    [255, 255, 255], # 1: White (Blood vessel)
])

def gpu_tensor(numpy_array):
    return torch.tensor(numpy_array, device=env.device)

def gpu_image_tensor_to_numpy_array(image_tensor):
    image = denormalize(image_tensor, config.channelwise_imagenet_mean_gpu_tensor, config.channelwise_imagenet_std_gpu_tensor)
    image = torch.clamp(image, 0, 1)
    image = image.permute(1, 2, 0).cpu().numpy()
    return (image * 255).astype(np.uint8)

def gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor):
    mask = mask_tensor.cpu().numpy()
    mask = np.clip(mask, 0, len(CLASS_COLORS) - 1).astype(np.int32)
    return CLASS_COLORS[mask]

def visualize_image(image_tensor):
    image = gpu_image_tensor_to_numpy_array(image_tensor)
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    plt.close()

def visualize_mask(mask_tensor):
    colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)
    plt.imshow(colored_mask)
    plt.axis('off')
    plt.show()
    plt.close()

def visualize_mask_overlayed_over_image(image_tensor, mask_tensor, alpha=0.5):
    image_array = gpu_image_tensor_to_numpy_array(image_tensor)
    colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)
    blended = (alpha * colored_mask + (1 - alpha) * image_array).astype(np.uint8)
    plt.imshow(blended)
    plt.axis('off')
    plt.show()
    plt.close()

def normalize(tensor, mean, std):
    return (tensor - mean) / std

def denormalize(tensor, mean, std):
    return tensor * std + mean

def print_model_torchinfo(model: nn.Module):
    print(torch_summary(model, input_size=(1, 3, config.image_width, config.image_height)))

def print_model(model: nn.Module):
    for name, module in model.named_modules():
        print(name, "->", module.__class__.__name__)

def create_dataloader(dataset, shuffle):
    return DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle, num_workers=config.num_workers, pin_memory=config.pin_memory, generator=config.generator)

def _num_batches(dataloader):
    return math.ceil(len(dataloader.dataset) / config.batch_size)

<h2 style = "color: green;">1. Data Exploration </h2>

In this section, you have to write your custom dataset class and visualize a few images (minimum five images) along with their mask overlayed.

<h2 style = "color: green;">1.1. Dataset Class [15 Points]</h2>

**In this sub-section, write your custom dataset class.**

**For example:**

```python
class SemSegDataset(Dataset):
    """ Generic Dataset class for semantic segmentation datasets.

        Arguments:
            train_image: path to input train images
            train_mask: path to input train mask labels
            train_tfms: transformations (if any)
            label_colors_list: mapping of label value to color code
            classes_to_train: number of classes

            Names of images in the images_folder and masks_folder should be the same for the same samples.
    """
```

In [ ]:
@dataclass
class SegmentationDataset(TorchDataset):
    images_root_folder: str
    masks_root_folder: str
    image_transforms: A.Compose
    image_filenames: np.ndarray
    has_masks: bool

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        image_filename = self.image_filenames[idx]
        image = cv2.cvtColor(cv2.imread(f'{self.images_root_folder}{image_filename}'), cv2.COLOR_BGR2RGB)
        if self.has_masks:
            mask_stem = Path(image_filename).stem
            mask_path = next(Path(self.masks_root_folder).glob(f'{mask_stem}.*'))
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            transformed = self.image_transforms(image=image, mask=mask)
            return transformed['image'], transformed['mask']
        transformed = self.image_transforms(image=image)
        return transformed['image']

<h2 style = "color: green;">1.2. Visualize dataset [10 Points]</h2>


**In this sub-section,  you have to plot a few images and its mask.**

**For example:**

---

<img src="https://www.dropbox.com/scl/fi/hrkxc7vlziawumy7pwfbt/grid_22.png?rlkey=d9jrybjc993qtbzngekj1sseu&st=7rsy2rik&dl=1">

---

In [ ]:
def plot_val_images_and_masks(num_images_to_visualize=3):
    val_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.val_transforms,
        image_filenames=env.fetch_val_filenames(),
        has_masks=True,
    )

    fig, axes = plt.subplots(num_images_to_visualize, 2, figsize=(10, 5 * num_images_to_visualize))
    axes[0, 0].set_title('Image')
    axes[0, 1].set_title('Mask')

    for i in range(num_images_to_visualize):
        image_tensor, mask_tensor = val_dataset[i]
        image = gpu_image_tensor_to_numpy_array(image_tensor)
        colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)

        axes[i, 0].imshow(image)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(colored_mask)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()

env = local_env
config = Config(training=False)
plot_val_images_and_masks()

<h2 style = "color: green;">2. Define Evaluation Metrics [10 Points]</h2>


This project is evaluated on the mean <a href='https://learnopencv.com/intersection-over-union-iou-in-object-detection-and-segmentation/'>IoU</a>.

Mean Intersection over Union (mIoU) is a metric that measures the average overlap between predicted and actual objects in an image or set of images. It's calculated by averaging the Intersection over Union (IoU) of each class in an image.

The following illustration helps in better understanding the IoU:

<img src = "https://www.dropbox.com/scl/fi/e67ri02rqou5oeazdh3rb/intersection-over-union-iou.jpg?rlkey=x5fi1ke6svoas4rh7mfxn3ro7&st=peahed63&dl=1" >

**In this section, you have to implement the mIoU evaluation metric.**

In [ ]:
def val_mean_iou(pred_masks):
    val_filenames = env.fetch_val_filenames()
    gt_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.test_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )

    # Accumulators for global mIoU (sum intersection/union across all images, then divide)
    global_intersection = np.zeros(2)
    global_union = np.zeros(2)

    # Per-image mIoU list
    ious_per_image = []

    for i in range(len(gt_dataset)):
        _, gt_mask = gt_dataset[i]
        gt_mask = gt_mask.numpy() if isinstance(gt_mask, torch.Tensor) else gt_mask
        pred_mask = pred_masks[i].cpu().numpy() if isinstance(pred_masks[i], torch.Tensor) else np.array(pred_masks[i])

        # Resize pred to GT dimensions if they differ
        if pred_mask.shape != gt_mask.shape:
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (gt_mask.shape[1], gt_mask.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)

        # Binarize both masks
        gt_binary = (gt_mask > 0).astype(np.uint8)
        pred_binary = (pred_mask > 0).astype(np.uint8)

        # Per-class IoU (background=0, vessel=1)
        class_ious = []
        for c in range(2):
            gt_c = (gt_binary == c)
            pred_c = (pred_binary == c)
            intersection = np.logical_and(gt_c, pred_c).sum()
            union = np.logical_or(gt_c, pred_c).sum()
            global_intersection[c] += intersection
            global_union[c] += union
            iou = intersection / (union + 1e-6)
            class_ious.append(iou)

        ious_per_image.append(np.mean(class_ious))

    per_image_miou = np.mean(ious_per_image)
    global_class_ious = global_intersection / (global_union + 1e-6)
    global_miou = np.mean(global_class_ious)

    print(f'Per-image mIoU: {per_image_miou:.4f}')
    print(f'Global mIoU:    {global_miou:.4f}')
    return global_miou

<h2 style = "color: green;">3. Model [10 Points]</h2>

**In this section, you have to define your model.**

<h2 style = "color: green;">4. Train & Inference</h2>

- **In this section, you have to train the model and infer on sample data.**


- **You can write your trainer class in this section.**


- **If you are using any loss function other than PyTorch standard loss function, you have to define in this section.**


- **This section should also have optimizer and LR-schedular (if using) details.**

<h2 style = "color: green;">4.1. Train [15 Points]</h2>


**Write your training code in this section.**


**This section must contain training plots (use matplotlib or share tensorboard scalars logs).**

**You must have to plot the following:**
- **train loss and validation loss**


- **train accuracy and validation accuracy**


- **Mean Intersection over Union (mIoU) of these classes on validation data.**

**An example of training plots is shown below:**

---

<img src='https://www.dropbox.com/scl/fi/c09gejfn8ivhapp2ngott/loss.png?rlkey=6pennpz1959plb84jlfjytw4p&st=e6ppi338&dl=1'>

---

<img src='https://www.dropbox.com/scl/fi/9h83bbzkifn2kon0gu3wi/accuracy.png?rlkey=v0q17rmi1djljny7ykr0ydk7b&st=zxze36ro&dl=1'>

---

<img src='https://www.dropbox.com/scl/fi/0pq7ivltoeyp1mlkv85oh/miou.png?rlkey=g4iuqvbwih9ld6efvayz5lgjj&st=5bpihwaz&dl=1'>



<h2 style = "color: green;">5. Inference [10 Points]</h2>

**Plot a few samples with the test image, predicted mask and the predicted mask overlayed onto the retinal eye image.**

**For example:**

---

<img src='https://www.dropbox.com/scl/fi/re6sw5od5k59u4bwgbfgy/prediction_overlay_segformer.png?rlkey=a34frfceiq2pawoovm4zx7b5r&st=qi3fivau&dl=1'>

---

